# Imports

##### standard python libraries

In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine
import netCDF4 as nc
from pprint import pprint

In [2]:
from pysdl import SDLClient

In [3]:
import warnings
warnings.filterwarnings('ignore')

##### Custom Libraries

In [4]:
import psql_utilities as psql_utl

### Set Paths

In [5]:
### path to configuration file of database
path_conf_file = './../../config/database.ini'
### path where to temporary store the file downloaded from SDL
path_tmp_download = '../../sample_data/download_tmp'

### Connect to SDL

In [6]:
client = SDLClient()
client.auth.login()

{'response': 200, 'message': 'Authenticated on SDL successfully.'}


{'message': 'User already attached as andrea.distefano@ingv.it.'}

### Database Connection

In [7]:
db_config = psql_utl.load_config(path_conf_file)
### connect to the database - sqlalchemy
engine = create_engine(db_config['db_uri'])

## Select Datasets based on Geospatial Info

Dataset entities may be characterized by a geospatial attribute, i.e. *spatial_coverage*.  Namely a bounding box (min_lon, min_lat, max_lon, max_lat) containing the coordinates of the dataset. One can use this metadata to retrieve the results of simulation runs in a given region of interest.

In order to design such queries one can:

 1. define a bounding box (*in_bbox*) identifying the region of interest
 2. select all datasets whose *spatial_coverage* intersects the bounding box of the region of interest. For simplicity in the metadata schema we have defined a custom function, called *intersect_2Dbboxes*, to determine whether two given bounding boxes intersect each other. This function takes as input two bounding boxes and returns a true/false depending on whether they intersect or not.
 3. filter datasets by selecting only Outputs Datasets stored in netCDF files

The full query SQL query can be found stored in *'./query_ds_output.sql'*, one should just specify bounding box of the region of interest. 

### setup

##### read query from file

In [8]:
query_ds = open("./query_ds_output.sql", "r")
query_ds = query_ds.read()
#query_ds

##### set input variables

In [9]:
# bounding box around Catania Area
bbox_ct = [15.0,37.2,15.15,37.6]
# bounding box around Siracusa Area
bbox_sr = [15.2,37.0,15.35,37.2]
# input bounding box
in_bbox = bbox_ct

### query result

##### define input parameters

In [10]:
dict_params = {'in_bbox': in_bbox}

##### run query with pandas

In [11]:
df_query = pd.read_sql(query_ds, con=engine,params=dict_params)
df_query

,ds_path,ds_name,ds_format,ds_type,ds_description,ds_spatial_coverage
0,CT-SR_M100/BS_001/E01267N3753E01646N3535-BS-M7...,C_CT.nc,nc,output,Output file of HySea simulation storing: the m...,"[15.021755834790268, 37.310341551192096, 15.12..."
1,CT-SR_M100/BS_001/E01267N3753E01646N3535-BS-M7...,grid0_ts.nc,nc,output,Tsunami time series at the specified points,"[15.0217558348, 36.9989732858, 15.3441088876, ..."
2,CT-SR_M100/BS_001/E01522N3585E03578N2685-BS-M7...,C_CT.nc,nc,output,Output file of HySea simulation storing: the m...,"[15.021755834790268, 37.310341551192096, 15.12..."
3,CT-SR_M100/BS_001/E01522N3585E03578N2685-BS-M7...,grid0_ts.nc,nc,output,Tsunami time series at the specified points,"[15.0217558348, 36.9989732858, 15.3441088876, ..."
4,CT-SR_M100/BS_001/E01267N3753E01646N3535-BS-M8...,C_CT.nc,nc,output,Output file of HySea simulation storing: the m...,"[15.021755834790268, 37.310341551192096, 15.12..."
...,...,...,...,...,...,...
963895,CT-SR_M100/PS_009/E02020N3739E02658N3366-PS-Mu...,grid0_ts.nc,nc,output,Tsunami time series at the specified points,"[15.0217558348, 36.9989732858, 15.3441088876, ..."
963896,CT-SR_M100/PS_009/E02020N3739E02658N3366-PS-Mu...,C_CT.nc,nc,output,Output file of HySea simulation storing: the m...,"[15.021755834790268, 37.310341551192096, 15.12..."
963897,CT-SR_M100/PS_009/E02020N3739E02658N3366-PS-Mu...,grid0_ts.nc,nc,output,Tsunami time series at the specified points,"[15.0217558348, 36.9989732858, 15.3441088876, ..."
963898,CT-SR_M100/PS_009/E02020N3739E02658N3366-PS-Mu...,C_CT.nc,nc,output,Output file of HySea simulation storing: the m...,"[15.021755834790268, 37.310341551192096, 15.12..."


**NOTE:** One can refine the results by further filtering the datasets according to their attributes. Then, we can use 'ds_path' to identify and access the corresponding file in the experiment folder (*path_experiment_dataset*).

### Access a File with SDL APIs

In [12]:
### select a dataset path
ds_path = df_query['ds_path'][0]
### download ds using SDL API
ds_label = ds_path.replace('/','__')
ds_outoput_path = os.path.join(path_tmp_download,ds_label)
print(f"Download File {ds_path} from SDL and save to {ds_outoput_path}")
client.file.download(experiment_id=207,version='1.0.0',file_path=ds_path,output_filename=ds_outoput_path)
## open file locally
ds_file = nc.Dataset(ds_outoput_path)
pprint(ds_file.variables)

Download File CT-SR_M100/BS_001/E01470N3877E01631N3641-BS-M680_E01579N3692_D010_S067D50R270_A000638_S009/grid0_ts.nc from SDL and save to ../../sample_data/download_tmp/CT-SR_M100__BS_001__E01470N3877E01631N3641-BS-M680_E01579N3692_D010_S067D50R270_A000638_S009__grid0_ts.nc


{
  "status": "ok",
  "message": "Download CT-SR_M100/BS_001/E01470N3877E01631N3641-BS-M680_E01579N3692_D010_S067D50R270_A000638_S009/grid0_ts.nc completed successfully!"
}

{'deformed_bathy': <class 'netCDF4.Variable'>
float32 deformed_bathy(grid_npoints)
    long_name: Grid deformed bathymetry
    standard_name: deformed depth
    units: meters
    missing_value: -9999.0
    _FillValue: -9999.0
unlimited dimensions: 
current shape = (8480,)
filling on,
 'eta': <class 'netCDF4.Variable'>
float32 eta(time, grid_npoints)
    units: meters
    long_name: Wave amplitude
    missing_value: -9999.0
    _FillValue: -9999.0
unlimited dimensions: time
current shape = (481, 8480)
filling on,
 'latitude': <class 'netCDF4.Variable'>
float64 latitude(grid_npoints)
    units: degrees
    missing_value: -9999.0
    _FillValue: -9999.0
unlimited dimensions: 
current shape = (8480,)
filling on,
 'longitude': <class 'netCDF4.Variable'>
float64 longitude(grid_npoints)
    units: degrees
    missing_value: -9999.0
    _FillValue: -9999.0
unlimited dimensions: 
current shape = (8480,)
filling on,
 'max_height': <class 'netCDF4.Variable'>
float32 max_height(grid_npoints)
    l

##### NOTE (access dataset locally): 

If the experiment files have already been downloaded, one can specify the path where the experiment is located (*path_experiment_dataset*) to directly open the files locally.